# Homework 10 - Online Learning Option1

https://scikit-learn.org/0.15/modules/scaling_strategies.html#incremental-learning

* Implement a mini batch functionality to train a regressor.
    - (Optional) If anyone want to do this in a pipeline can do this: https://koaning.github.io/tokenwiser/api/pipeline.html

* Save model, load the model again and test it on `X_test` __Do NOT commit the pickle file__

In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [19]:
def test_df():
    df = pd.read_csv('https://raw.githubusercontent.com/msaricaumbc/DS_data/master/ds602/car_prices/car_prices.csv', low_memory=False)

    df = df.sample(5000, random_state=100).reset_index(drop=True)
    
    y = df['sellingprice']
    df.drop('sellingprice', axis=1, inplace=True)
    X = df
    
    return X,y

def partial_df():
    df = pd.read_csv('https://raw.githubusercontent.com/msaricaumbc/DS_data/master/ds602/car_prices/car_prices.csv', low_memory=False)
   
    while(True):
        yield df.sample(100).reset_index(drop=True)
        
gen = partial_df()

In [20]:
X_test, y_test = test_df()

In [21]:
# each time you call this you will get a new slice of the dataframe.
next(gen)

,year,make,model,trim,body,transmission,vin,state,condition,odometer,color,interior,seller,mmr,sellingprice,saledate
0,2013,Chevrolet,Impala,LS Fleet,Sedan,automatic,2g1wf5e3xd1147731,fl,3,63190.0,gray,gray,the hertz corporation,9000,8500,Tue Jan 06 2015 01:30:00 GMT-0800 (PST)
1,2008,Chrysler,Town and Country,Touring,Minivan,automatic,2a8hr54p58r649493,fl,3.2,160146.0,white,gray,enterprise vehicle exchange / rental / tulsa,4050,3100,Fri Jan 23 2015 01:25:00 GMT-0800 (PST)
2,2014,Hyundai,Genesis Coupe,3.8 Ultimate,Genesis Coupe,manual,kmhhu6kj9eu117622,nc,4.6,7618.0,black,black,remarketing by ge/the credit union loan source,25700,25500,Tue Feb 17 2015 01:15:00 GMT-0800 (PST)
3,2000,Chevrolet,Monte Carlo,SS,Coupe,automatic,2g1wx12k9y9315609,va,3.6,174531.0,red,black,toyota of stafford,1450,2400,Thu Dec 18 2014 09:35:00 GMT-0800 (PST)
4,2007,Chevrolet,Tahoe,LT,suv,automatic,1gnfk13077r334982,mn,3.9,167800.0,gold,black,luther brookdale chevrolet,10600,13100,Wed Jun 17 2015 03:30:00 GMT-0700 (PDT)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2012,Toyota,Camry,LE,sedan,automatic,4t1bf1fk1cu584701,tn,4.1,35285.0,black,gray,world omni financial corporation,14750,16300,Wed Jun 17 2015 03:30:00 GMT-0700 (PDT)
96,2012,MINI,Cooper,Base,hatchback,NaN,wmwsu3c59ct540162,fl,4.2,19362.0,red,—,bmw mini financial services,12050,12000,Wed Jun 17 2015 02:40:00 GMT-0700 (PDT)
97,2014,Chevrolet,Malibu,LTZ,Sedan,automatic,1g11h5sl8ef202681,sc,2,24671.0,silver,black,enterprise vehicle exchange / tra / rental / t...,17250,10600,Thu Jan 29 2015 04:00:00 GMT-0800 (PST)
98,2011,Toyota,Tundra,Tundra FFV,Double Cab,automatic,5tfuw5f12bx174756,mo,4.3,91643.0,—,gray,"auto plaza motor sports, llc",19150,19600,Tue Jan 06 2015 03:00:00 GMT-0800 (PST)


In [22]:
#extracting features and target from one training the mini-batch
batch = next(gen)

y_batch = batch['sellingprice']
X_batch = batch.drop('sellingprice', axis=1)

print("X_batch shape:", X_batch.shape)
print("y_batch shape:", y_batch.shape)

X_batch shape: (100, 15)
y_batch shape: (100,)


In [23]:
# defining numeric and categorical feature groups correctly
numeric_features = ['year', 'odometer']

categorical_features = [
    'make', 'model', 'trim', 'body',
    'transmission', 'state', 'color',
    'interior', 'condition', 'mmr'
]

In [24]:
# building preprocessing steps for numeric and categorical columns
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

In [25]:
# fitting the preprocessor on the first batch and converting it into model-ready format
first_batch = next(gen)

y_first = first_batch['sellingprice']
X_first = first_batch.drop('sellingprice', axis=1)

X_first_processed = preprocessor.fit_transform(X_first)

print("Processed first batch shape:", X_first_processed.shape)

Processed first batch shape: (100, 338)


In [26]:
# creating an SGD regressor that can learn incrementally using mini-batches
model = SGDRegressor(
    max_iter=1,
    tol=None,
    random_state=42
)

In [27]:
# training the model on the first processed mini-batch
model.partial_fit(X_first_processed, y_first)

print("First batch trained successfully.")

First batch trained successfully.


In [28]:
# updating the model repeatedly with new mini-batches from the generator
num_batches = 100

for i in range(num_batches):
    batch = next(gen)

    y_batch = batch['sellingprice']
    X_batch = batch.drop('sellingprice', axis=1)

    X_batch_processed = preprocessor.transform(X_batch)

    model.partial_fit(X_batch_processed, y_batch)

print("Mini-batch training completed.")

Mini-batch training completed.


In [29]:
# combining preprocessing and model into one pipeline and predicting directly
full_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('regressor', model)
])

y_pred = full_pipeline.predict(X_test)

print("Number of predictions:", len(y_pred))

Number of predictions: 5000


In [30]:
# evaluating the model using regression metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

MAE: 4092.2483831525474
RMSE: 6297.598283749623
R2: 0.5743695950633336


the model achieved an R² score of about 0.55 showing that it explains a moderate portion of the variation in car selling prices the MAE and RMSE values show reasonable prediction error levels for this dataset indicating that incremental learning with mini-batches can successfully train a regression model on streaming-style data.

In [31]:
# saving the trained pipeline so it can be reused later
joblib.dump(full_pipeline, "online_learning_pipeline.pkl")

print("Pipeline saved successfully.")

Pipeline saved successfully.


In [32]:
# loading the saved pipeline back into memory
loaded_pipeline = joblib.load("online_learning_pipeline.pkl")

print("Pipeline loaded successfully.")

Pipeline loaded successfully.


In [33]:
# testing the loaded pipeline again on X_test to confirm it still works
y_pred_loaded = loaded_pipeline.predict(X_test)

print("Loaded pipeline predictions:", len(y_pred_loaded))

Loaded pipeline predictions: 5000


EXPLANATION

in this assignment i implemented an online learning regression model using SGDRegressor to predict car selling prices instead of training the model on the entire dataset at once i used incremental learning with mini-batches generated from the partial_df() generator which provided small subsets of the data repeatedly a preprocessing pipeline was created using a ColumnTransformer to handle missing values scale numeric features and encode categorical variables so the model could process mixed data types correctly the model was updated iteratively using partial_fit() across multiple mini-batches simulating a streaming data learning scenario after training the model was evaluated on a fixed test dataset using MAE RMSE and R² where the results showed reasonable predictive performance for this type of dataset hence the complete pipeline was saved using joblib reloaded successfully and tested again on X_test to confirm that the saved model maintained consistent prediction behavior.